In [4]:
!pip install -q ultralytics #lap onnxruntime onnxslim


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.8 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[

In [59]:
import shutil

shutil.rmtree("/kaggle/working/models")

In [1]:
import shutil

src = "/kaggle/input/models/romanresner/uncertainty-aware-crop-perception/pytorch/default/15/models" 
dst = "/kaggle/working/models"



shutil.copytree(src, dst)

'/kaggle/working/models'

In [6]:
yaml_content = """
path: /kaggle/input/notebooks/romanresner/phenobench-yolo-dataset/dataset_yolo


train: images/train
val: images/val
test: images/test

names:
  0: crop
  1: weed

task: segment
"""

with open("phenobench.yaml", "w") as f:
    f.write(yaml_content)

print("phenobench.yaml erstellt")

phenobench.yaml erstellt


In [63]:
from ultralytics import YOLO
import torch
import gc

# Speicher vor dem Start bereinigen
gc.collect()
torch.cuda.empty_cache()

# --- Gemeinsame Einstellungen ------------------------------------------
DATA  = "/kaggle/working/phenobench.yaml"
IMGSZ = 960
BATCH = 4  # Sollte es wieder zu OOM-Fehlern kommen, bitte auf 2 oder 1 senken

# --- 1. FP32 Baseline (.pt) --------------------------------------------
print("=" * 50)
print("1. FP32 Baseline (.pt)")
print("=" * 50)
model_pt = YOLO("/kaggle/working/models/phenobench_cropweed_seg_yolo11s_960.pt", task="segment")
metrics_pt = model_pt.val(data=DATA, imgsz=IMGSZ, split="val", device=0, batch=BATCH)

# --- 2. FP32 Baseline (.onnx) ------------------------------------------
print("=" * 50)
print("2. FP32 Baseline (.onnx)")
print("=" * 50)
model_onnx = YOLO("/kaggle/working/models/phenobench_cropweed_seg_yolo11s_960.onnx", task="segment")
metrics_onnx = model_onnx.val(data=DATA, imgsz=IMGSZ, split="val", device=0, batch=BATCH)

# --- 3. FP32 TensorRT (.engine) ----------------------------------------
print("=" * 50)
print("3. FP32 TensorRT (.engine)")
print("=" * 50)
model_trt = YOLO("/kaggle/working/models/phenobench_cropweed_seg_yolo11s_960.engine", task="segment")
metrics_trt = model_trt.val(data=DATA, imgsz=IMGSZ, split="val", device=0, batch=BATCH)

# --- 4. Vergleich ------------------------------------------------------
print("\n" + "=" * 73)
print("VERGLEICH: PyTorch vs ONNX vs TensorRT (Alle FP32)")
print("=" * 73)
print(f"{'Metrik':<15} {'PT (FP32)':>11} {'ONNX (FP32)':>12} {'TRT (FP32)':>12} {'Delta (ONNX)':>12} {'Delta (TRT)':>11}")
print("-" * 73)

metrics = [
    ("Box mAP50",    metrics_pt.box.map50,  metrics_onnx.box.map50,  metrics_trt.box.map50),
    ("Box mAP50-95", metrics_pt.box.map,    metrics_onnx.box.map,    metrics_trt.box.map),
    ("Seg mAP50",    metrics_pt.seg.map50,  metrics_onnx.seg.map50,  metrics_trt.seg.map50),
    ("Seg mAP50-95", metrics_pt.seg.map,    metrics_onnx.seg.map,    metrics_trt.seg.map),
]

for name, pt, onnx, trt in metrics:
    delta_onnx = onnx - pt
    delta_trt = trt - pt
    print(f"{name:<15} {pt:>11.4f} {onnx:>12.4f} {trt:>12.4f} {delta_onnx:>+12.4f} {delta_trt:>+11.4f}")

print("=" * 73)
print("Hinweis: Minimale Abweichungen bei ONNX/TensorRT im Nachkommastellenbereich")
print("sind aufgrund von hardware- und compilerspezifischen Optimierungen normal.")

1. FP32 Baseline (.pt)
Ultralytics 8.4.59 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s-seg summary (fused): 114 layers, 10,067,590 parameters, 0 gradients, 32.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1948.6±546.9 MB/s, size: 2603.0 KB)
val: Scanning /kaggle/input/notebooks/romanresner/phenobench-yolo-dataset/dataset_yolo/labels/val... 772 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 772/772 114.5it/s 6.7s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/notebooks/romanresner/phenobench-yolo-dataset/dataset_yolo/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 193/193 6.0it/s 32.4s0.2ss
                   all        772      10389      0.878      0.816      0.871      0.673      0.873      0.787      0.845      0.562
                  crop        772       6470      0.951      0.919      0.958

In [1]:
import shutil

src = "/kaggle/input/models/romanresner/uncertainty-aware-crop-perception/pytorch/default/15" 
dst = "/kaggle/working/project"

shutil.copytree(src, dst)

'/kaggle/working/project'

In [9]:
import yaml
import os
import json
import subprocess

# --- 1. Define Paths and Settings --------------------------------------
PROJECT_DIR = "/kaggle/working/project"
CONFIG_PATH = "/kaggle/working/project/configs/default.yaml"

# Models (paths relative to the project directory or absolute)
models = [
    "models/phenobench_cropweed_seg_yolo11s_960.pt",
    "models/phenobench_cropweed_seg_yolo11s_960.onnx",
    "models/phenobench_cropweed_seg_yolo11s_960.engine"
]

fps_results = {}
latency_results = {}

# --- 2. Loop Over All 3 Models -----------------------------------------
for model_path in models:
    # Extract file extension (pt, onnx, or engine)
    model_ext = model_path.split(".")[-1]
    
    print("=" * 83)
    print(f"Configuring and starting model: {model_ext.upper()}")
    print("=" * 83)
    
    # Read the configuration file (YAML)
    with open(CONFIG_PATH, "r") as f:
        data = yaml.safe_load(f)
    
    # Dynamically update YAML configuration values
    data["model"]["path"] = model_path
    data["io"]["input_video"] = "/kaggle/working/project/vertical_drone_flight.mp4"
    data["io"]["output_video"] = f"/kaggle/working/output/vertical_advanced_agricultural_{model_ext}.mp4"
    data["io"]["metrics_output_csv"] = f"/kaggle/working/output/data/{model_ext}_run_metrics.csv"
    data["io"]["metrics_output_json"] = f"/kaggle/working/output/data/{model_ext}_metrics.json"
    
    data["io"]["max_frames"] = -1
    data["io"]["side_by_side"] = True
    
    # Save the configuration file
    with open(CONFIG_PATH, "w") as f:
        yaml.dump(data, f, sort_keys=False)
    
    # Change to the project directory (equivalent to %cd)
    os.chdir(PROJECT_DIR)
    
    # Run main.py (equivalent to !python main.py)
    # check=True ensures that the script stops if any error occurs
    subprocess.run(["python", "main.py"], check=True)
    
    # Read performance metrics from the newly created JSON file
    json_path = f"/kaggle/working/output/data/{model_ext}_metrics.json"
    if os.path.exists(json_path):
        try:
            with open(json_path, "r") as f:
                summary = json.load(f)
            fps_results[model_ext] = summary["performance_metrics"]["mean_fps"]
            latency_results[model_ext] = summary["performance_metrics"]["mean_inference_time_ms"]
        except Exception as e:
            print(f"Error reading JSON file for {model_ext}: {e}")
            fps_results[model_ext] = 0.0
            latency_results[model_ext] = 0.0
    else:
        print(f"Warning: JSON file {json_path} was not found.")
        fps_results[model_ext] = 0.0
        latency_results[model_ext] = 0.0

# --- 3. Final Comparison Output of Performance Metrics -----------------
print("\n" + "=" * 83)
print("COMPARISON: Video Inference (All FP32)")
print("=" * 83)
print(f"{'Metric':<15} {'PT (FP32)':>12} {'ONNX (FP32)':>13} {'TRT (FP32)':>13} {'Delta (ONNX)':>13} {'Delta (TRT)':>12}")
print("-" * 83)

# Output average FPS values
fps_pt = fps_results.get("pt", 0.0)
fps_onnx = fps_results.get("onnx", 0.0)
fps_trt = fps_results.get("engine", 0.0)

d_fps_onnx = fps_onnx - fps_pt
d_fps_trt = fps_trt - fps_pt

print(f"{'Avg FPS':<15} {fps_pt:>12.1f} {fps_onnx:>13.1f} {fps_trt:>13.1f} {d_fps_onnx:>+13.1f} {d_fps_trt:>+12.1f}")

# Output latency values (ms)
lat_pt = latency_results.get("pt", 0.0)
lat_onnx = latency_results.get("onnx", 0.0)
lat_trt = latency_results.get("engine", 0.0)

d_lat_onnx = lat_onnx - lat_pt
d_lat_trt = lat_trt - lat_pt

print(f"{'Latency (ms)':<15} {lat_pt:>12.2f} {lat_onnx:>13.2f} {lat_trt:>13.2f} {d_lat_onnx:>+13.2f} {d_lat_trt:>+12.2f}")

print("=" * 83)
print("Note: Minimal variations for ONNX/TensorRT in decimal places are normal")
print("due to hardware- and compiler-specific optimizations.")

Configuring and starting model: PT
MAIN CUDA: True
MAIN GPU COUNT: 1
PYTHON: /usr/bin/python3
Starting Industrial Perception Pipeline...
Info: NVML successfully initialized for GPU telemetry.
TRACKER CUDA: True
TRACKER GPU COUNT: 1
TRACKER DEVICE: 0
Using TensorRT engine: models/phenobench_cropweed_seg_yolo11s_960.engine
Loading model: models/phenobench_cropweed_seg_yolo11s_960.engine
Tracker initialized using: bytetrack.yaml
Processing full video: 708 frames.
Loading models/phenobench_cropweed_seg_yolo11s_960.engine for TensorRT inference...
requirements: Ultralytics requirement ['tensorrt-cu12>=7.0.0,!=10.2.0'] not found, attempting AutoUpdate...


  0%|          | 0/708 [00:00<?, ?it/s]

Using Python 3.12.13 environment at: /usr
Resolved 3 packages in 31.01s
Prepared 3 packages in 22.47s
Installed 3 packages in 54ms
 + tensorrt-cu12==11.0.0.114
 + tensorrt-cu12-bindings==11.0.0.114
 + tensorrt-cu12-libs==11.0.0.114

requirements: AutoUpdate success ✅ 54.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

[06/01/2026-18:09:56] [TRT] [I] Loaded engine size: 48 MiB
[06/01/2026-18:09:56] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +97, now: CPU 0, GPU 148 (MiB)


100%|██████████| 708/708 [02:52<00:00,  4.11it/s] 


📊 JSON Summary successfully compiled and saved to: /kaggle/working/output/data/pt_run_metrics.json

Resources released.
Pipeline execution completed successfully.
Configuring and starting model: ONNX
MAIN CUDA: True
MAIN GPU COUNT: 1
PYTHON: /usr/bin/python3
Starting Industrial Perception Pipeline...
Info: NVML successfully initialized for GPU telemetry.
TRACKER CUDA: True
TRACKER GPU COUNT: 1
TRACKER DEVICE: 0
Using TensorRT engine: models/phenobench_cropweed_seg_yolo11s_960.onnx
Loading model: models/phenobench_cropweed_seg_yolo11s_960.onnx
Tracker initialized using: bytetrack.yaml
Processing full video: 708 frames.
Loading models/phenobench_cropweed_seg_yolo11s_960.onnx for ONNX Runtime inference...
requirements: Ultralytics requirement ['onnxruntime-gpu'] not found, attempting AutoUpdate...


...

📊 JSON Summary successfully compiled and saved to: /kaggle/working/output/data/onnx_run_metrics.json

Resources released.
Pipeline execution completed successfully.
Configuring and starting model: ENGINE
MAIN CUDA: True
MAIN GPU COUNT: 1
PYTHON: /usr/bin/python3
Starting Industrial Perception Pipeline...
Info: NVML successfully initialized for GPU telemetry.
TRACKER CUDA: True
TRACKER GPU COUNT: 1
TRACKER DEVICE: 0
Using TensorRT engine: models/phenobench_cropweed_seg_yolo11s_960.engine
Loading model: models/phenobench_cropweed_seg_yolo11s_960.engine
Tracker initialized using: bytetrack.yaml
Processing full video: 708 frames.
Loading models/phenobench_cropweed_seg_yolo11s_960.engine for TensorRT inference...


  0%|          | 0/708 [00:00<?, ?it/s]

[06/01/2026-18:14:21] [TRT] [I] Loaded engine size: 48 MiB
[06/01/2026-18:14:21] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +97, now: CPU 0, GPU 148 (MiB)


100%|██████████| 708/708 [01:55<00:00,  6.12it/s]


📊 JSON Summary successfully compiled and saved to: /kaggle/working/output/data/engine_run_metrics.json

Resources released.
Pipeline execution completed successfully.

COMPARISON: Video Inference (All FP32)
Metric             PT (FP32)   ONNX (FP32)    TRT (FP32)  Delta (ONNX)  Delta (TRT)
-----------------------------------------------------------------------------------
Avg FPS                  0.0           0.0           0.0          +0.0         +0.0
Latency (ms)            0.00          0.00          0.00         +0.00        +0.00
Note: Minimal variations for ONNX/TensorRT in decimal places are normal
due to hardware- and compiler-specific optimizations.


In [ ]:
#%cd /kaggle/working/project
#!python main.py

In [12]:
import yaml
import os
import csv
import subprocess

# --- 1. Define Paths and Settings --------------------------------------
PROJECT_DIR = "/kaggle/working/project"
CONFIG_PATH = "/kaggle/working/project/configs/default.yaml"

# Models (paths relative to the project directory or absolute)
models = [
    "models/phenobench_cropweed_seg_yolo11s_960.pt",
    "models/phenobench_cropweed_seg_yolo11s_960.onnx",
    "models/phenobench_cropweed_seg_yolo11s_960.engine"
]

fps_results = {}
latency_results = {}


def get_metrics_from_csv(csv_path):
    """
    Reads the performance CSV file and calculates the average FPS
    and mean inference latency.
    """
    if not os.path.exists(csv_path):
        return 0.0, 0.0
    
    try:
        inference_times = []
        frame_times = []
        
        with open(csv_path, mode='r', newline='') as f:
            reader = csv.DictReader(f)
            for row in reader:
                inference_times.append(float(row["inference_time_ms"]))
                frame_times.append(float(row["frame_time_ms"]))
        
        if not inference_times or not frame_times:
            return 0.0, 0.0
        
        # Calculate mean values from CSV rows
        mean_inference_time = sum(inference_times) / len(inference_times)
        mean_frame_time = sum(frame_times) / len(frame_times)
        mean_fps = 1000.0 / mean_frame_time if mean_frame_time > 0 else 0.0
        
        return mean_fps, mean_inference_time
    except Exception as e:
        print(f"Error reading CSV file {csv_path}: {e}")
        return 0.0, 0.0


# --- 2. Loop Over All 3 Models -----------------------------------------
for model_path in models:
    # Extract file extension (pt, onnx, or engine)
    model_ext = model_path.split(".")[-1]
    
    print("=" * 83)
    print(f"Configuring and starting model: {model_ext.upper()}")
    print("=" * 83)
    
    # Read the configuration file (YAML)
    with open(CONFIG_PATH, "r") as f:
        data = yaml.safe_load(f)
    
    # Dynamically update YAML configuration values
    data["model"]["path"] = model_path
    data["io"]["input_video"] = "/kaggle/working/project/vertical_drone_flight.mp4"
    data["io"]["output_video"] = f"/kaggle/working/output/vertical_advanced_agricultural_{model_ext}.mp4"
    data["io"]["metrics_output_csv"] = f"/kaggle/working/output/data/{model_ext}_run_metrics.csv"
    data["io"]["metrics_output_json"] = f"/kaggle/working/output/data/{model_ext}_metrics.json"
    
    data["io"]["max_frames"] = -1
    data["io"]["side_by_side"] = True
    
    # Save the configuration file
    with open(CONFIG_PATH, "w") as f:
        yaml.dump(data, f, sort_keys=False)
    
    # Change to the project directory (equivalent to %cd)
    os.chdir(PROJECT_DIR)
    
    # Run main.py (equivalent to !python main.py)
    # check=True ensures that the script stops if any error occurs
    subprocess.run(["python", "main.py"], check=True)
    
    # Calculate performance metrics directly from the newly created CSV file
    csv_path = f"/kaggle/working/output/data/{model_ext}_run_metrics.csv"
    mean_fps, mean_latency = get_metrics_from_csv(csv_path)
    
    fps_results[model_ext] = mean_fps
    latency_results[model_ext] = mean_latency

# --- 3. Final Comparison Output of Performance Metrics -----------------
print("\n" + "=" * 83)
print("COMPARISON: Video Inference (All FP32)")
print("=" * 83)
print(f"{'Metric':<15} {'PT (FP32)':>12} {'ONNX (FP32)':>13} {'TRT (FP32)':>13} {'Delta (ONNX)':>13} {'Delta (TRT)':>12}")
print("-" * 83)

# Output average FPS values
fps_pt = fps_results.get("pt", 0.0)
fps_onnx = fps_results.get("onnx", 0.0)
fps_trt = fps_results.get("engine", 0.0)

d_fps_onnx = fps_onnx - fps_pt
d_fps_trt = fps_trt - fps_pt

print(f"{'Avg FPS':<15} {fps_pt:>12.1f} {fps_onnx:>13.1f} {fps_trt:>13.1f} {d_fps_onnx:>+13.1f} {d_fps_trt:>+12.1f}")

# Output latency values (ms)
lat_pt = latency_results.get("pt", 0.0)
lat_onnx = latency_results.get("onnx", 0.0)
lat_trt = latency_results.get("engine", 0.0)

d_lat_onnx = lat_onnx - lat_pt
d_lat_trt = lat_trt - lat_pt

print(f"{'Latency (ms)':<15} {lat_pt:>12.2f} {lat_onnx:>13.2f} {lat_trt:>13.2f} {d_lat_onnx:>+13.2f} {d_lat_trt:>+12.2f}")

print("=" * 83)
print("Note: Minimal variations for ONNX/TensorRT in decimal places are normal")
print("due to hardware- and compiler-specific optimizations.")

Configuring and starting model: PT
MAIN CUDA: True
MAIN GPU COUNT: 1
PYTHON: /usr/bin/python3
Starting Industrial Perception Pipeline...
Info: NVML successfully initialized for GPU telemetry.
TRACKER CUDA: True
TRACKER GPU COUNT: 1
TRACKER DEVICE: 0
Using TensorRT engine: models/phenobench_cropweed_seg_yolo11s_960.engine
Loading model: models/phenobench_cropweed_seg_yolo11s_960.engine
Tracker initialized using: bytetrack.yaml
Processing full video: 708 frames.
Loading models/phenobench_cropweed_seg_yolo11s_960.engine for TensorRT inference...


  0%|          | 0/708 [00:00<?, ?it/s]

[06/01/2026-18:25:38] [TRT] [I] Loaded engine size: 48 MiB
[06/01/2026-18:25:38] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +97, now: CPU 0, GPU 148 (MiB)


100%|██████████| 708/708 [01:59<00:00,  5.92it/s]


📊 JSON Summary successfully compiled and saved to: /kaggle/working/output/data/pt_run_metrics.json

Resources released.
Pipeline execution completed successfully.
Configuring and starting model: ONNX
MAIN CUDA: True
MAIN GPU COUNT: 1
PYTHON: /usr/bin/python3
Starting Industrial Perception Pipeline...
Info: NVML successfully initialized for GPU telemetry.
TRACKER CUDA: True
TRACKER GPU COUNT: 1
TRACKER DEVICE: 0
Using TensorRT engine: models/phenobench_cropweed_seg_yolo11s_960.onnx
Loading model: models/phenobench_cropweed_seg_yolo11s_960.onnx
Tracker initialized using: bytetrack.yaml
Processing full video: 708 frames.
Loading models/phenobench_cropweed_seg_yolo11s_960.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.26.0 with CUDAExecutionProvider


...

📊 JSON Summary successfully compiled and saved to: /kaggle/working/output/data/onnx_run_metrics.json

Resources released.
Pipeline execution completed successfully.
Configuring and starting model: ENGINE
MAIN CUDA: True
MAIN GPU COUNT: 1
PYTHON: /usr/bin/python3
Starting Industrial Perception Pipeline...
Info: NVML successfully initialized for GPU telemetry.
TRACKER CUDA: True
TRACKER GPU COUNT: 1
TRACKER DEVICE: 0
Using TensorRT engine: models/phenobench_cropweed_seg_yolo11s_960.engine
Loading model: models/phenobench_cropweed_seg_yolo11s_960.engine
Tracker initialized using: bytetrack.yaml
Processing full video: 708 frames.
Loading models/phenobench_cropweed_seg_yolo11s_960.engine for TensorRT inference...


  0%|          | 0/708 [00:00<?, ?it/s]

[06/01/2026-18:29:59] [TRT] [I] Loaded engine size: 48 MiB
[06/01/2026-18:29:59] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +97, now: CPU 0, GPU 148 (MiB)


100%|██████████| 708/708 [01:50<00:00,  6.43it/s]


📊 JSON Summary successfully compiled and saved to: /kaggle/working/output/data/engine_run_metrics.json

Resources released.
Pipeline execution completed successfully.

COMPARISON: Video Inference (All FP32)
Metric             PT (FP32)   ONNX (FP32)    TRT (FP32)  Delta (ONNX)  Delta (TRT)
-----------------------------------------------------------------------------------
Avg FPS                 13.7          10.7          14.7          -3.0         +1.0
Latency (ms)           69.65         90.01         65.17        +20.36        -4.49
Note: Minimal variations for ONNX/TensorRT in decimal places are normal
due to hardware- and compiler-specific optimizations.


In [1]:

import shutil

src = "/kaggle/input/models/romanresner/uncertainty-aware-crop-perception/pytorch/default/17" 
dst = "/kaggle/working/project"

shutil.copytree(src, dst)

'/kaggle/working/project'

In [2]:
import yaml

path = "/kaggle/working/project/configs/default.yaml"

with open(path, "r") as f:
    data = yaml.safe_load(f)

data["model"]["path"] = "models/phenobench_cropweed_seg_yolo11s_960.engine"
data["io"]["input_video"] = "/kaggle/working/project/vertical_drone_flight.mp4"

data["io"]["output_video"] = "/kaggle/working/output/vertical_advanced_agricultural_engine.mp4"

data["io"]["metrics_output_csv"] = "/kaggle/working/output/run_metrics.csv"

data["io"]["metrics_output_json"] = "/kaggle/working/output/data/metrics.json"

data["io"]["max_frames"] = 100
data["io"]["side_by_side"] = True

with open(path, "w") as f:
    yaml.dump(data, f, sort_keys=False)

In [5]:
%cd /kaggle/working/project
!python main_profile.py

/kaggle/working/project
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PROFILING RUN
MAIN CUDA: True
MAIN GPU COUNT: 2
PYTHON: /usr/bin/python3
Starting Industrial Perception Pipeline with Profiling...
Info: NVML successfully initialized for GPU telemetry.
TRACKER CUDA: True
TRACKER GPU COUNT: 2
TRACKER DEVICE: 0
Loading model: models/phenobench_cropweed_seg_yolo11s_960.engine
Tracker initialized using: bytetrack.yaml
Processing full video: 708 frames.
  0%|                                                   | 0/708 [00:00<?, ?it/s]requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 200ms
Prepared 1 package in 35ms
Installed 1